In [2]:
import pandas as pd
import numpy as np

In [3]:
df1=pd.read_excel(r'C:\Users\samae\Downloads\Syllabus\depi\data\SalesData2025_Item_Cust.xlsx')

In [4]:
item_map=pd.read_excel(r'C:\Users\samae\Downloads\Syllabus\depi\data\item map id.xlsx')
item_map=item_map.set_index('ItemID Map')

In [ ]:
import pandas as pd
df1['Date'] = pd.to_datetime(df1['Date'])

# خليه بدون الساعات عشان يبقى بس اليوم
df1['Date'] = df1['Date'].dt.normalize()

df1['Month'] = df1['Date'].dt.to_period('M')

# Aggregate daily demand per product
daily_demand = df1.groupby(['Date', 'ItemID-'])['Qty'].sum().reset_index()

# Aggregate monthly demand per product
monthly_demand = df1.groupby(['Month', 'ItemID-'])['Qty'].sum().reset_index()

In [6]:
daily_demand

,Date,ItemID-,Qty
0,2025-01-01,BCH-3-08-00272,4
1,2025-01-01,BCH-3-08-00273,76
2,2025-01-01,BCH-3-08-00274,163
3,2025-01-01,BCH-3-08-00275,106
4,2025-01-01,BCH-3-08-00276,17
...,...,...,...
16280,2025-06-27,MRM-4-01-00062,2
16281,2025-06-27,NKA-2-05-00151,9
16282,2025-06-27,SNX-2-06-00086,1
16283,2025-06-27,TQT-2-10-00043,1


In [7]:
# Merge Item info
daily_demand = daily_demand.merge(item_map, left_on='ItemID-',right_on='ItemID Map',how='left')
monthly_demand = monthly_demand.merge(item_map,left_on='ItemID-',right_on='ItemID Map', how='left')

In [8]:
# لو Monthly
monthly_demand['Avg_Daily_Demand'] = monthly_demand['Qty'] / 30  # أو عدد أيام الشهر

# Safety Stock 20% (كمثال)
monthly_demand['Safety_Stock'] = monthly_demand['Avg_Daily_Demand'] * 0.2* monthly_demand['Factor'] #daily safety stock to calculate rop

In [9]:
Lead_Time = 7  # أيام مثلا
monthly_demand['ROP'] = monthly_demand['Avg_Daily_Demand'] * Lead_Time + monthly_demand['Safety_Stock']

In [10]:
monthly_demand

,Month,ItemID-,Qty,ItemName-Map,BrandID-Map,BrandName-Map,Master Brand ID - Map,MasterBrandName-Map,LargeUOM,Factor,Avg_Daily_Demand,Safety_Stock,ROP
0,2025-01,BCH-3-08-00272,122,بوشارتي ملح,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,4.066667,2.44,30.906667
1,2025-01,BCH-3-08-00273,30088,بوشارتي ملح باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1002.933333,601.76,7622.293333
2,2025-01,BCH-3-08-00274,45880,بوشارتي هوت باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1529.333333,917.60,11622.933333
3,2025-01,BCH-3-08-00275,30676,بوشارتي هوت باربكيو نيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1022.533333,613.52,7771.253333
4,2025-01,BCH-3-08-00276,1534,مقرمش كاتشب كلاسيك,BCH-MQR02,مقرمش,BCH,مقرمشات بوشارتي,PL,3.0,51.133333,30.68,388.613333
...,...,...,...,...,...,...,...,...,...,...,...,...,...
881,2025-06,TQT-2-01-00478,2022,طقطقة بوب كورن كلاسيك,TQT01,طقطقة,TQT,طقطقة,PL,18.0,67.400000,242.64,714.440000
882,2025-06,TQT-2-10-00014,1,طقطقة شيبس فواكه,TQT01,طقطقة,TQT,طقطقة,PL,6.0,0.033333,0.04,0.273333
883,2025-06,TQT-2-10-00042,3039,طقطقة هوت سوبر,TQT01,طقطقة,TQT,طقطقة,PL,6.0,101.300000,121.56,830.660000
884,2025-06,TQT-2-10-00043,4941,طقطقة جبنة كوول,TQT01,طقطقة,TQT,طقطقة,PL,6.0,164.700000,197.64,1350.540000


In [11]:
# من daily_demand نعمل weekly demand
daily_demand['Week'] = daily_demand['Date'].dt.isocalendar().week

weekly_demand = daily_demand.groupby(['Week', 'ItemID-'])['Qty'].sum().reset_index()

In [12]:
weekly_demand

,Week,ItemID-,Qty
0,1,BCH-3-08-00272,20
1,1,BCH-3-08-00273,936
2,1,BCH-3-08-00274,1660
3,1,BCH-3-08-00275,1175
4,1,BCH-3-08-00276,104
...,...,...,...
3316,26,SNX-2-06-00087,97
3317,26,TQT-2-01-00478,540
3318,26,TQT-2-10-00042,1982
3319,26,TQT-2-10-00043,1376


In [13]:
daily_demand['Year'] = daily_demand['Date'].dt.year
# 2️⃣ Aggregate الطلب على مستوى الأسبوع + المنتج
weekly_demand = daily_demand.groupby(['Year','Week','ItemID-'])['Qty'].sum().reset_index()

# 3️⃣ ندمج مع بيانات المنتج (Item dimension)
weekly_demand = weekly_demand.merge(item_map, left_on='ItemID-',right_on='ItemID Map', how='left')

# 4️⃣ نحسب Avg Daily Demand لكل أسبوع
weekly_demand['Avg_Daily_Demand'] = weekly_demand['Qty'] / 7  # 7 أيام في الأسبوع

# 5️⃣ نحسب Safety Stock (مثال 20%)
weekly_demand['Safety_Stock'] = weekly_demand['Avg_Daily_Demand'] * 0.2 * weekly_demand['Factor']

# 6️⃣ نحسب ROP (Reorder Point) لو Lead Time = 7 أيام مثلا
Lead_Time = 7
weekly_demand['ROP'] = weekly_demand['Avg_Daily_Demand'] * Lead_Time + weekly_demand['Safety_Stock']


In [14]:
weekly_demand

,Year,Week,ItemID-,Qty,ItemName-Map,BrandID-Map,BrandName-Map,Master Brand ID - Map,MasterBrandName-Map,LargeUOM,Factor,Avg_Daily_Demand,Safety_Stock,ROP
0,2025,1,BCH-3-08-00272,20,بوشارتي ملح,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,2.857143,1.714286,21.714286
1,2025,1,BCH-3-08-00273,936,بوشارتي ملح باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,133.714286,80.228571,1016.228571
2,2025,1,BCH-3-08-00274,1660,بوشارتي هوت باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,237.142857,142.285714,1802.285714
3,2025,1,BCH-3-08-00275,1175,بوشارتي هوت باربكيو نيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,167.857143,100.714286,1275.714286
4,2025,1,BCH-3-08-00276,104,مقرمش كاتشب كلاسيك,BCH-MQR02,مقرمش,BCH,مقرمشات بوشارتي,PL,3.0,14.857143,8.914286,112.914286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3316,2025,26,SNX-2-06-00087,97,كِسرة جبنة باربكيو,SNX01,كِسرة,SNX,سناكس,PL,24.0,13.857143,66.514286,163.514286
3317,2025,26,TQT-2-01-00478,540,طقطقة بوب كورن كلاسيك,TQT01,طقطقة,TQT,طقطقة,PL,18.0,77.142857,277.714286,817.714286
3318,2025,26,TQT-2-10-00042,1982,طقطقة هوت سوبر,TQT01,طقطقة,TQT,طقطقة,PL,6.0,283.142857,339.771429,2321.771429
3319,2025,26,TQT-2-10-00043,1376,طقطقة جبنة كوول,TQT01,طقطقة,TQT,طقطقة,PL,6.0,196.571429,235.885714,1611.885714


In [15]:
daily= df1.groupby(['Date','ItemID-']).agg(
    Avg_UnitPrice = ('UnitPrice','mean'),
    Total_Promo = ('PromotionsTotal','sum'),
    Total_CashDiscount = ('CashDiscount','sum'),
    Total_ManualDiscount = ('ManualDiscount','sum'),
    Total_Taxes = ('TaxesTotal','sum'),
    Total_Revenue = ('LineTotal','sum')
).reset_index()

In [16]:
daily_demand=daily_demand.merge(daily,on=['Date','ItemID-'],how='outer')

In [17]:
daily_demand

,Date,ItemID-,Qty,ItemName-Map,BrandID-Map,BrandName-Map,Master Brand ID - Map,MasterBrandName-Map,LargeUOM,Factor,Week,Year,Avg_UnitPrice,Total_Promo,Total_CashDiscount,Total_ManualDiscount,Total_Taxes,Total_Revenue
0,2025-01-01,BCH-3-08-00272,4,بوشارتي ملح,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1,2025,658.630000,0,0,0,368.84,3003.36
1,2025-01-01,BCH-3-08-00273,76,بوشارتي ملح باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1,2025,662.811795,-257,0,0,7405.74,60303.69
2,2025-01-01,BCH-3-08-00274,163,بوشارتي هوت باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1,2025,571.941410,-328,0,0,14084.15,114684.79
3,2025-01-01,BCH-3-08-00275,106,بوشارتي هوت باربكيو نيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1,2025,746.749836,-283,0,0,11302.96,92039.11
4,2025-01-01,BCH-3-08-00276,17,مقرمش كاتشب كلاسيك,BCH-MQR02,مقرمش,BCH,مقرمشات بوشارتي,PL,3.0,1,2025,637.693333,-49,0,0,1509.12,12288.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16280,2025-06-27,MRM-4-01-00062,2,مرشملو جبنة هوت,MRM-01,مرشملو,MRM,مرشملو,CT,192.0,26,2025,98.020000,-16,0,0,25.30,205.96
16281,2025-06-27,NKA-2-05-00151,9,نكهة ريحان,NKA01,نكهة,NKA,نكهة,PL,36.0,26,2025,278.032000,-60,0,0,341.01,2776.78
16282,2025-06-27,SNX-2-06-00086,1,كِسرة جبنة كلاسيك,SNX01,كِسرة,SNX,سناكس,PL,24.0,26,2025,332.060000,-7,0,0,45.56,370.98
16283,2025-06-27,TQT-2-10-00043,1,طقطقة جبنة كوول,TQT01,طقطقة,TQT,طقطقة,PL,6.0,26,2025,379.440000,-11,0,0,51.53,419.59


In [18]:
df1['Week'] = df1['Date'].dt.to_period('W')
weekly = df1.groupby(['Week','ItemID-']).agg(
    Avg_UnitPrice=('UnitPrice','mean'),
    Total_Promo=('PromotionsTotal','sum'),
    Total_CashDiscount=('CashDiscount','sum'),
    Total_ManualDiscount=('ManualDiscount','sum'),
    Total_Taxes=('TaxesTotal','sum'),
    Total_Revenue=('LineTotal','sum')
).reset_index()
weekly['Week'] = weekly['Week'].apply(lambda x: x.week if hasattr(x, 'week') else x)

In [19]:
weekly_demand=weekly_demand.merge(weekly,on=['Week','ItemID-'],how='outer')

In [20]:
weekly_demand

,Year,Week,ItemID-,Qty,ItemName-Map,BrandID-Map,BrandName-Map,Master Brand ID - Map,MasterBrandName-Map,LargeUOM,Factor,Avg_Daily_Demand,Safety_Stock,ROP,Avg_UnitPrice,Total_Promo,Total_CashDiscount,Total_ManualDiscount,Total_Taxes,Total_Revenue
0,2025,1,BCH-3-08-00272,20,بوشارتي ملح,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,2.857143,1.714286,21.714286,557.929231,-13,0,0,1655.35,1.347916e+04
1,2025,1,BCH-3-08-00273,936,بوشارتي ملح باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,133.714286,80.228571,1016.228571,649.535750,-2254,0,0,91489.91,7.449884e+05
2,2025,1,BCH-3-08-00274,1660,بوشارتي هوت باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,237.142857,142.285714,1802.285714,633.052979,-4305,0,0,161971.09,1.318905e+06
3,2025,1,BCH-3-08-00275,1175,بوشارتي هوت باربكيو نيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,167.857143,100.714286,1275.714286,743.634000,-3510,0,0,132863.89,1.081895e+06
4,2025,1,BCH-3-08-00276,104,مقرمش كاتشب كلاسيك,BCH-MQR02,مقرمش,BCH,مقرمشات بوشارتي,PL,3.0,14.857143,8.914286,112.914286,616.606610,-392,0,0,9051.89,7.370877e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3316,2025,26,SNX-2-06-00087,97,كِسرة جبنة باربكيو,SNX01,كِسرة,SNX,سناكس,PL,24.0,13.857143,66.514286,163.514286,341.366562,-557,0,0,4555.80,3.709684e+04
3317,2025,26,TQT-2-01-00478,540,طقطقة بوب كورن كلاسيك,TQT01,طقطقة,TQT,طقطقة,PL,18.0,77.142857,277.714286,817.714286,213.847981,-1719,0,0,10939.45,8.907860e+04
3318,2025,26,TQT-2-10-00042,1982,طقطقة هوت سوبر,TQT01,طقطقة,TQT,طقطقة,PL,6.0,283.142857,339.771429,2321.771429,254.369940,-25314,0,0,72396.78,5.895171e+05
3319,2025,26,TQT-2-10-00043,1376,طقطقة جبنة كوول,TQT01,طقطقة,TQT,طقطقة,PL,6.0,196.571429,235.885714,1611.885714,370.555256,-14360,0,0,70656.75,5.753461e+05


In [21]:
monthly = df1.groupby(['Month','ItemID-']).agg(
    Avg_UnitPrice=('UnitPrice','mean'),
    Total_Promo=('PromotionsTotal','sum'),
    Total_CashDiscount=('CashDiscount','sum'),
    Total_ManualDiscount=('ManualDiscount','sum'),
    Total_Taxes=('TaxesTotal','sum'),
    Total_Revenue=('LineTotal','sum')
).reset_index()

In [22]:
monthly_demand=monthly_demand.merge(monthly,on=['Month','ItemID-'],how='outer')

In [23]:
monthly_demand

,Month,ItemID-,Qty,ItemName-Map,BrandID-Map,BrandName-Map,Master Brand ID - Map,MasterBrandName-Map,LargeUOM,Factor,Avg_Daily_Demand,Safety_Stock,ROP,Avg_UnitPrice,Total_Promo,Total_CashDiscount,Total_ManualDiscount,Total_Taxes,Total_Revenue
0,2025-01,BCH-3-08-00272,122,بوشارتي ملح,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,4.066667,2.44,30.906667,585.533667,-798,0,0,10794.34,8.789643e+04
1,2025-01,BCH-3-08-00273,30088,بوشارتي ملح باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1002.933333,601.76,7622.293333,678.188841,-42111,0,0,2869404.51,2.336515e+07
2,2025-01,BCH-3-08-00274,45880,بوشارتي هوت باربكيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1529.333333,917.60,11622.933333,653.185346,-61661,0,0,4160477.91,3.387817e+07
3,2025-01,BCH-3-08-00275,30676,بوشارتي هوت باربكيو نيو,BCH-BCH01,بوشارتي,BCH,مقرمشات بوشارتي,PL,3.0,1022.533333,613.52,7771.253333,760.885393,-49610,0,0,3287205.75,2.676728e+07
4,2025-01,BCH-3-08-00276,1534,مقرمش كاتشب كلاسيك,BCH-MQR02,مقرمش,BCH,مقرمشات بوشارتي,PL,3.0,51.133333,30.68,388.613333,594.405561,-3314,0,0,131683.32,1.072284e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
881,2025-06,TQT-2-01-00478,2022,طقطقة بوب كورن كلاسيك,TQT01,طقطقة,TQT,طقطقة,PL,18.0,67.400000,242.64,714.440000,264.891117,-8215,0,0,49259.48,4.011136e+05
882,2025-06,TQT-2-10-00014,1,طقطقة شيبس فواكه,TQT01,طقطقة,TQT,طقطقة,PL,6.0,0.033333,0.04,0.273333,46.920000,0,0,0,6.57,5.349000e+01
883,2025-06,TQT-2-10-00042,3039,طقطقة هوت سوبر,TQT01,طقطقة,TQT,طقطقة,PL,6.0,101.300000,121.56,830.660000,255.396929,-32309,0,0,109390.94,8.907563e+05
884,2025-06,TQT-2-10-00043,4941,طقطقة جبنة كوول,TQT01,طقطقة,TQT,طقطقة,PL,6.0,164.700000,197.64,1350.540000,371.439023,-52026,0,0,252376.86,2.055058e+06


In [24]:
daily_demand.to_csv(r'C:\Users\samae\Downloads\Syllabus\depi\data\daily.csv', index=False, encoding='utf-8-sig')

In [25]:
weekly_demand.to_csv(r'C:\Users\samae\Downloads\Syllabus\depi\data\weekly.csv', index=False, encoding='utf-8-sig')